
# Generic Health Insurance EDA & Pattern-Based Rule Mining

Reusable for claims fraud, leakage, denial, high-cost claims, provider abuse, churn, lapse, grievance escalation, underwriting risk, utilization anomalies, and operational delays.

The notebook performs file loading, data profiling, missing-value handling, feature typing, health-insurance feature engineering, univariate/bivariate EDA, numeric binning, exhaustive combination mining, FP-Growth association rules, decision-tree rules, holdout validation, risk bands, and Excel/CSV export.

> Discovered rules are associations, not proof of causation. Review for leakage, stability, fairness, regulatory acceptability, and operational usefulness before deployment.


## 1. Install and import libraries

In [ ]:

%pip install -q pandas numpy openpyxl scikit-learn matplotlib mlxtend scipy


In [ ]:

import os, re, math, warnings
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, export_text, _tree
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
from mlxtend.frequent_patterns import fpgrowth, association_rules
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 300)


## 2. Configuration

In [ ]:

FILE_PATH = 'your_uploaded_file.xlsx'
SHEET_NAME = 0

# Supported target modes: binary, one_vs_rest, multiclass, ordinal
TARGET_MODE = 'multiclass'
TARGET_COLUMN = 'risk_level'
POSITIVE_CLASS = 'Y'
FOCUS_CLASS = 'High'
TARGET_CLASSES = ['Low','Medium','High']
TARGET_ORDER = {'Low':1,'Medium':2,'High':3}

ID_COLUMNS=[]; DATE_COLUMNS=[]; FREE_TEXT_COLUMNS=[]
FORCE_CATEGORICAL=[]; FORCE_NUMERIC=[]; EXCLUDE_COLUMNS=[]
HEALTH_FIELD_MAP={'claim_amount':None,'premium_amount':None,'sum_insured':None,'policy_start_date':None,'claim_date':None,'admission_date':None,'discharge_date':None,'member_age':None,'room_rent':None}
MIN_SUPPORT_COUNT=20; MIN_CLASS_COUNT=5; MIN_CONFIDENCE=.35; MIN_LIFT=1.3
MAX_RULE_LENGTH=3; TOP_N_CATEGORIES=20; MAX_NUMERIC_BINS=5
ASSOCIATION_MIN_SUPPORT=.02; ASSOCIATION_MIN_CONFIDENCE=.35; ASSOCIATION_MIN_LIFT=1.3; ASSOCIATION_MAX_LEN=4
TREE_MAX_DEPTH=5; TREE_MIN_SAMPLES_LEAF=20; TEST_SIZE=.25; RANDOM_STATE=42
OUTPUT_FOLDER='pattern_mining_outputs'; OUTPUT_EXCEL='health_insurance_pattern_rules.xlsx'
os.makedirs(OUTPUT_FOLDER,exist_ok=True)


## 3. Load and standardize data

In [ ]:

def load_data(path, sheet_name=0):
    ext = os.path.splitext(path)[1].lower()
    if ext in ['.xlsx', '.xls']:
        return pd.read_excel(path, sheet_name=sheet_name)
    if ext == '.csv':
        return pd.read_csv(path)
    if ext in ['.parquet', '.pq']:
        return pd.read_parquet(path)
    raise ValueError(f'Unsupported file type: {ext}')

def clean_name(x):
    return re.sub(r'[^a-z0-9]+', '_', str(x).strip().lower()).strip('_')

df_raw = load_data(FILE_PATH, SHEET_NAME)
df = df_raw.copy()
df.columns = [clean_name(c) for c in df.columns]
TARGET_COLUMN = clean_name(TARGET_COLUMN)
ID_COLUMNS = [clean_name(c) for c in ID_COLUMNS]
DATE_COLUMNS = [clean_name(c) for c in DATE_COLUMNS]
FREE_TEXT_COLUMNS = [clean_name(c) for c in FREE_TEXT_COLUMNS]
FORCE_CATEGORICAL = [clean_name(c) for c in FORCE_CATEGORICAL]
FORCE_NUMERIC = [clean_name(c) for c in FORCE_NUMERIC]
EXCLUDE_COLUMNS = [clean_name(c) for c in EXCLUDE_COLUMNS]
for k,v in list(HEALTH_FIELD_MAP.items()):
    if v: HEALTH_FIELD_MAP[k] = clean_name(v)

df = df.replace(['?', '', 'NA', 'N/A', 'NULL', 'null', 'None', 'none', '-', '--'], np.nan)
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].astype('string').str.strip().replace({'': pd.NA})
for c in DATE_COLUMNS:
    if c in df.columns: df[c] = pd.to_datetime(df[c], errors='coerce')
print('Shape:', df.shape)
display(df.head())


## 4. Data profile and target preparation

In [ ]:

def data_profile(data):
    rows=[]
    for c in data.columns:
        s=data[c]
        rows.append({'column':c,'dtype':str(s.dtype),'rows':len(s),'non_null':int(s.notna().sum()),'missing':int(s.isna().sum()),'missing_pct':round(s.isna().mean()*100,2),'unique':int(s.nunique(dropna=True)),'sample_values':', '.join(map(str,s.dropna().astype(str).unique()[:5]))})
    return pd.DataFrame(rows)
profile=data_profile(df)
display(profile)

for c in DATE_COLUMNS:
    if c in df.columns: df[c]=pd.to_datetime(df[c],errors='coerce')
if TARGET_COLUMN not in df.columns: raise KeyError(f"Target column {TARGET_COLUMN} not found")

def norm(x): return str(x).strip().lower()
df=df[df[TARGET_COLUMN].notna()].copy()
if TARGET_MODE=='binary':
    df['target_flag']=(df[TARGET_COLUMN].map(norm)==norm(POSITIVE_CLASS)).astype(int)
elif TARGET_MODE=='one_vs_rest':
    df['target_flag']=(df[TARGET_COLUMN].map(norm)==norm(FOCUS_CLASS)).astype(int)
elif TARGET_MODE=='multiclass':
    df['target_class']=df[TARGET_COLUMN].astype(str).str.strip()
    classes=[str(x) for x in TARGET_CLASSES] if TARGET_CLASSES else sorted(df['target_class'].unique())
    df['target_class']=pd.Categorical(df['target_class'],categories=classes)
elif TARGET_MODE=='ordinal':
    mapping={norm(k):v for k,v in TARGET_ORDER.items()}
    df['target_class']=df[TARGET_COLUMN].astype(str).str.strip()
    df['target_ordinal']=df[TARGET_COLUMN].map(lambda x:mapping.get(norm(x),np.nan))
    df=df[df['target_ordinal'].notna()].copy()
else: raise ValueError('Invalid TARGET_MODE')

print('Mode:',TARGET_MODE,'Rows:',len(df))
if TARGET_MODE in ['binary','one_vs_rest']:
    print('Positive rate:',df['target_flag'].mean())
else:
    display(df['target_class'].value_counts(normalize=True).rename('class_rate').to_frame())


## 5. Feature typing and health-insurance features

In [ ]:

def infer_types(data):
    excluded=set([TARGET_COLUMN,'target_flag','target_class','target_ordinal']+ID_COLUMNS+FREE_TEXT_COLUMNS+EXCLUDE_COLUMNS)
    num=[]; cat=[]; dates=[]
    for c in [x for x in data.columns if x not in excluded]:
        if c in DATE_COLUMNS or pd.api.types.is_datetime64_any_dtype(data[c]): dates.append(c)
        elif c in FORCE_CATEGORICAL: cat.append(c)
        elif c in FORCE_NUMERIC or pd.api.types.is_numeric_dtype(data[c]): num.append(c)
        else: cat.append(c)
    return num,cat,dates

def safe_ratio(a,b): return a/b.replace(0,np.nan)
def add_health_features(data,f):
    out=data.copy()
    def ok(k): return f.get(k) and f[k] in out.columns
    if ok('claim_amount') and ok('premium_amount'):
        out['claim_to_premium_ratio']=safe_ratio(pd.to_numeric(out[f['claim_amount']],errors='coerce'),pd.to_numeric(out[f['premium_amount']],errors='coerce'))
    if ok('claim_amount') and ok('sum_insured'):
        out['claim_to_sum_insured_ratio']=safe_ratio(pd.to_numeric(out[f['claim_amount']],errors='coerce'),pd.to_numeric(out[f['sum_insured']],errors='coerce'))
    if ok('policy_start_date') and ok('claim_date'):
        out['policy_tenure_days_at_claim']=(pd.to_datetime(out[f['claim_date']],errors='coerce')-pd.to_datetime(out[f['policy_start_date']],errors='coerce')).dt.days
    if ok('admission_date') and ok('discharge_date'):
        out['calculated_length_of_stay']=(pd.to_datetime(out[f['discharge_date']],errors='coerce')-pd.to_datetime(out[f['admission_date']],errors='coerce')).dt.days
    if ok('room_rent') and ok('claim_amount'):
        out['room_rent_share']=safe_ratio(pd.to_numeric(out[f['room_rent']],errors='coerce'),pd.to_numeric(out[f['claim_amount']],errors='coerce'))
    if ok('member_age'):
        age=pd.to_numeric(out[f['member_age']],errors='coerce')
        out['member_age_band']=pd.cut(age,[-np.inf,18,30,45,60,75,np.inf],labels=['<=18','19-30','31-45','46-60','61-75','76+']).astype('string')
    return out

df=add_health_features(df,HEALTH_FIELD_MAP)
numeric_cols,categorical_cols,date_cols=infer_types(df)
for c in numeric_cols+categorical_cols:
    if df[c].isna().any(): df[f'{c}__missing']=df[c].isna().astype(int)
numeric_cols,categorical_cols,date_cols=infer_types(df)
print('Numeric:',len(numeric_cols),'Categorical:',len(categorical_cols),'Dates:',len(date_cols))


## 6. Numeric and categorical EDA

In [ ]:

def numeric_eda(data,cols):
    rows=[]; grp='target_flag' if TARGET_MODE in ['binary','one_vs_rest'] else 'target_class'
    for c in cols:
        s=pd.to_numeric(data[c],errors='coerce')
        for cls,idxs in data.groupby(grp,observed=False).groups.items():
            x=s.loc[idxs]
            rows.append({'feature':c,'target_class':str(cls),'count':int(x.notna().sum()),'missing':int(x.isna().sum()),'mean':x.mean(),'median':x.median(),'std':x.std(),'min':x.min(),'p25':x.quantile(.25),'p75':x.quantile(.75),'max':x.max()})
    return pd.DataFrame(rows)
numeric_summary=numeric_eda(df,numeric_cols)
display(numeric_summary.head(50))

def categorical_lift(data,feature):
    temp=data[[feature]].copy(); temp[feature]=temp[feature].astype('string').fillna('UNKNOWN')
    if TARGET_MODE in ['binary','one_vs_rest']:
        temp['target']=data['target_flag'].values
        out=temp.groupby(feature)['target'].agg(records='count',class_count='sum').reset_index()
        out['target_class']=str(POSITIVE_CLASS if TARGET_MODE=='binary' else FOCUS_CLASS)
        out['class_rate']=out.class_count/out.records; out['global_class_rate']=data.target_flag.mean()
    else:
        temp['target_class']=data['target_class'].astype(str).values
        out=temp.groupby([feature,'target_class']).size().rename('class_count').reset_index()
        sup=temp.groupby(feature).size().rename('records').reset_index(); out=out.merge(sup,on=feature)
        rates=temp.target_class.value_counts(normalize=True).to_dict(); out['class_rate']=out.class_count/out.records; out['global_class_rate']=out.target_class.map(rates)
    out['lift']=out.class_rate/out.global_class_rate.clip(lower=1e-9); out['feature']=feature; out['value']=out[feature].astype(str)
    return out[['feature','value','target_class','records','class_count','class_rate','global_class_rate','lift']]
univariate_rules=pd.concat([categorical_lift(df,c) for c in categorical_cols if df[c].nunique(dropna=True)<=500],ignore_index=True) if categorical_cols else pd.DataFrame()
if not univariate_rules.empty: univariate_rules=univariate_rules.sort_values(['lift','records'],ascending=False)
display(univariate_rules.head(150))


## 7. Prepare rule-mining data

In [ ]:

def discretize(data,cols,max_bins=5):
    out=pd.DataFrame(index=data.index);meta=[]
    for c in cols:
        s=pd.to_numeric(data[c],errors='coerce')
        if s.nunique(dropna=True)<2:continue
        try:
            b,e=pd.qcut(s,q=min(max_bins,s.nunique()),duplicates='drop',retbins=True)
            out[f'{c}__bin']=b.astype('string').fillna('UNKNOWN');meta.append({'feature':c,'edges':e.tolist()})
        except Exception:
            out[f'{c}__bin']=pd.cut(s,bins=max_bins,duplicates='drop').astype('string').fillna('UNKNOWN')
    return out,pd.DataFrame(meta)

numeric_binned,bin_metadata=discretize(df,numeric_cols,MAX_NUMERIC_BINS)
rule_frame=pd.DataFrame(index=df.index)
for c in categorical_cols:
    v=df[c].astype('string').fillna('UNKNOWN')
    if v.nunique()>TOP_N_CATEGORIES:
        top=v.value_counts().head(TOP_N_CATEGORIES).index;v=v.where(v.isin(top),'OTHER')
    rule_frame[c]=v
rule_frame=pd.concat([rule_frame,numeric_binned],axis=1)
rule_frame['target_flag']=df.target_flag.values
print(rule_frame.shape)


## 8. Exhaustive combination mining

In [ ]:

def mine_combinations(data,max_len=3,min_support=20,min_pos=5,min_conf=.35,min_lift=1.3):
    results=[]; excluded={'target_flag','target_class','target_ordinal'}; features=[c for c in data.columns if c not in excluded]
    if TARGET_MODE in ['binary','one_vs_rest']:
        global_rates={str(POSITIVE_CLASS if TARGET_MODE=='binary' else FOCUS_CLASS):data.target_flag.mean()}
    else: global_rates=data.target_class.value_counts(normalize=True).to_dict()
    for L in range(1,max_len+1):
        for cols in combinations(features,L):
            if TARGET_MODE in ['binary','one_vs_rest']:
                g=data.groupby(list(cols),dropna=False).target_flag.agg(records='count',class_count='sum').reset_index(); g['target_class']=list(global_rates)[0]; g['global_class_rate']=list(global_rates.values())[0]
            else:
                g=data.groupby(list(cols),dropna=False).target_class.value_counts().rename('class_count').reset_index(); s=data.groupby(list(cols),dropna=False).size().rename('records').reset_index(); g=g.merge(s,on=list(cols)); g['global_class_rate']=g.target_class.map(global_rates)
            g['class_rate']=g.class_count/g.records; g['lift']=g.class_rate/g.global_class_rate.clip(lower=1e-9)
            g=g[(g.records>=min_support)&(g.class_count>=min_pos)&(g.class_rate>=min_conf)&(g.lift>=min_lift)]
            for _,r in g.iterrows():
                results.append({'rule_length':L,'rule':' AND '.join(f'{c} = {r[c]}' for c in cols),'features':' | '.join(cols),'target_class':str(r.target_class),'records':int(r.records),'class_count':int(r.class_count),'class_rate':float(r.class_rate),'global_class_rate':float(r.global_class_rate),'lift':float(r.lift)})
    out=pd.DataFrame(results)
    if not out.empty:
        out['rule_score']=out.lift*out.class_rate*np.log1p(out.records)*np.sqrt(out.class_count); out=out.sort_values(['rule_score','lift','records'],ascending=False)
    return out
combination_rules=mine_combinations(rule_frame,MAX_RULE_LENGTH,MIN_SUPPORT_COUNT,MIN_CLASS_COUNT,MIN_CONFIDENCE,MIN_LIFT)
display(combination_rules.head(150))


## 9. FP-Growth association rules

In [ ]:

parts=[]
for c in [x for x in rule_frame.columns if x not in {'target_flag','target_class','target_ordinal'}]:
    parts.append(pd.get_dummies(rule_frame[c].astype(str).map(lambda v:f'{c}={v}')).astype(bool))
basket=pd.concat(parts,axis=1)
if TARGET_MODE in ['binary','one_vs_rest']:
    name=f"TARGET={POSITIVE_CLASS if TARGET_MODE=='binary' else FOCUS_CLASS}"; basket[name]=rule_frame.target_flag.astype(bool).values; target_items={name}
else:
    td=pd.get_dummies(rule_frame.target_class.astype(str).map(lambda v:f'TARGET={v}')).astype(bool); basket=pd.concat([basket,td],axis=1); target_items=set(td.columns)
fi=fpgrowth(basket,min_support=ASSOCIATION_MIN_SUPPORT,use_colnames=True,max_len=ASSOCIATION_MAX_LEN)
rules=association_rules(fi,metric='confidence',min_threshold=ASSOCIATION_MIN_CONFIDENCE)
rules=rules[rules.consequents.apply(lambda x:len(x)==1 and next(iter(x)) in target_items)]
rules=rules[rules.lift>=ASSOCIATION_MIN_LIFT].copy(); rules['target_class']=rules.consequents.apply(lambda x:next(iter(x)).replace('TARGET=','')); rules['rule']=rules.antecedents.apply(lambda x:' AND '.join(sorted(x)))+' => TARGET='+rules.target_class
association_rules_positive=rules.sort_values(['lift','confidence','support'],ascending=False)
display(association_rules_positive[['rule','target_class','support','confidence','lift']].head(150))


## 10. Decision-tree model and rule extraction

In [ ]:

model_features=[c for c in df.columns if c not in set([TARGET_COLUMN,'target_flag','target_class','target_ordinal']+ID_COLUMNS+FREE_TEXT_COLUMNS+EXCLUDE_COLUMNS+date_cols)]
X=df[model_features].copy();y=df.target_flag.copy()
num=[c for c in model_features if pd.api.types.is_numeric_dtype(X[c])];cat=[c for c in model_features if c not in num]
prep=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median'))]),num),
                        ('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('enc',OneHotEncoder(handle_unknown='ignore',min_frequency=5))]),cat)])
model=DecisionTreeClassifier(max_depth=TREE_MAX_DEPTH,min_samples_leaf=TREE_MIN_SAMPLES_LEAF,class_weight='balanced',random_state=RANDOM_STATE)
pipe=Pipeline([('prep',prep),('model',model)])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=TEST_SIZE,stratify=y,random_state=RANDOM_STATE)
pipe.fit(X_train,y_train);pred=pipe.predict(X_test);proba=pipe.predict_proba(X_test)[:,1]
print(classification_report(y_test,pred));print('ROC-AUC',roc_auc_score(y_test,proba),'PR-AUC',average_precision_score(y_test,proba));print(confusion_matrix(y_test,pred))
feature_names=pipe.named_steps['prep'].get_feature_names_out()
print(export_text(pipe.named_steps['model'],feature_names=list(feature_names),decimals=3))

def extract_rules(tree,names):
    t=tree.tree_;out=[]
    def rec(node,conds):
        if t.feature[node]!=_tree.TREE_UNDEFINED:
            name=names[t.feature[node]];thr=t.threshold[node]
            rec(t.children_left[node],conds+[f'{name} <= {thr:.4f}']);rec(t.children_right[node],conds+[f'{name} > {thr:.4f}'])
        else:
            counts=t.value[node][0];total=counts.sum();pos=counts[1] if len(counts)>1 else 0
            out.append({'rule':' AND '.join(conds),'weighted_records':float(total),'weighted_positives':float(pos),'predicted_positive_rate':float(pos/total if total else 0),'predicted_class':int(np.argmax(counts))})
    rec(0,[]);return pd.DataFrame(out).sort_values(['predicted_positive_rate','weighted_records'],ascending=False)
tree_rules=extract_rules(pipe.named_steps['model'],list(feature_names));display(tree_rules)


## 11. Holdout validation and risk bands

In [ ]:

if TARGET_MODE in ['binary','one_vs_rest']:
    scored=pd.DataFrame({'actual':y_test.values,'score':proba[:,1]}); scored['risk_band']=pd.qcut(scored.score,q=10,duplicates='drop')
    risk_band_summary=scored.groupby('risk_band',observed=False).agg(records=('actual','size'),class_count=('actual','sum'),average_score=('score','mean')).reset_index(); risk_band_summary['class_rate']=risk_band_summary.class_count/risk_band_summary.records
else:
    classes=[str(c) for c in tree_pipe.named_steps['model'].classes_]; rows=[]
    for i,c in enumerate(classes):
        a=pd.Series(y_test.astype(str).values).eq(c); p=pd.Series(pred.astype(str)).eq(c)
        rows.append({'target_class':c,'actual_records':int(a.sum()),'predicted_records':int(p.sum()),'correct_predictions':int((a&p).sum()),'class_recall':((a&p).sum()/a.sum()) if a.sum() else np.nan,'class_precision':((a&p).sum()/p.sum()) if p.sum() else np.nan,'average_probability':float(proba[:,i].mean())})
    risk_band_summary=pd.DataFrame(rows)
display(risk_band_summary)


## 12. Final business rules and export

In [ ]:

final_rules=combination_rules.copy()
if not final_rules.empty:
    final_rules['recommended_action']=np.where(final_rules.lift>=2,'Enhanced validation / specialist review',np.where(final_rules.lift>=1.5,'Additional automated checks','Monitor'))
    final_rules['rule_id']=[f'R{i:04d}' for i in range(1,len(final_rules)+1)]

ordinal_rule_summary=pd.DataFrame()
if TARGET_MODE=='ordinal':
    rows=[]; features=[c for c in rule_frame.columns if c not in {'target_flag','target_class','target_ordinal'}]; portfolio_avg=rule_frame.target_ordinal.mean()
    for L in range(1,MAX_RULE_LENGTH+1):
        for cols in combinations(features,L):
            g=rule_frame.groupby(list(cols),dropna=False).target_ordinal.agg(records='size',average_severity='mean',median_severity='median').reset_index(); g=g[g.records>=MIN_SUPPORT_COUNT]
            for _,r in g.iterrows(): rows.append({'rule_length':L,'rule':' AND '.join(f'{c} = {r[c]}' for c in cols),'records':int(r.records),'average_severity':float(r.average_severity),'median_severity':float(r.median_severity),'severity_index':float(r.average_severity/portfolio_avg)})
    ordinal_rule_summary=pd.DataFrame(rows).sort_values(['severity_index','records'],ascending=False)

display(final_rules.head(150)); display(ordinal_rule_summary.head(150))
excel_path=os.path.join(OUTPUT_FOLDER,OUTPUT_EXCEL)
with pd.ExcelWriter(excel_path,engine='openpyxl') as w:
    profile.to_excel(w,sheet_name='Data_Profile',index=False); numeric_summary.to_excel(w,sheet_name='Numeric_EDA',index=False); univariate_rules.to_excel(w,sheet_name='Univariate_Lift',index=False); combination_rules.to_excel(w,sheet_name='Combination_Rules',index=False); risk_band_summary.to_excel(w,sheet_name='Model_Performance',index=False); final_rules.to_excel(w,sheet_name='Final_Business_Rules',index=False)
    if not association_rules_positive.empty: association_rules_positive.to_excel(w,sheet_name='Association_Rules',index=False)
    if not ordinal_rule_summary.empty: ordinal_rule_summary.to_excel(w,sheet_name='Ordinal_Severity_Rules',index=False)
print('Created',excel_path)



## 13. Operational interpretation

Prioritize rules with adequate support, high lift, narrow confidence intervals, acceptable holdout performance, low train-to-test degradation, clear business meaning, and no post-outcome leakage.

Typical health-insurance patterns include short policy tenure plus high claim-to-premium ratio, non-network provider plus high room-rent share, repeated provider/procedure combinations, missing clinical documentation plus high-cost claims, overlapping admissions, diagnosis/procedure incompatibility, and unusual utilization close to policy expiry.
